# Video inference on test.mp4

Run cells top to bottom. The notebook reads a trained YOLO model, runs `test.mp4`, saves an annotated video, and writes all detections to CSV.

In [3]:
from pathlib import Path
import time

import cv2
import pandas as pd
import torch
import yaml
from ultralytics import YOLO


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "scripts").exists():
            return path
    raise FileNotFoundError("Project root with pyproject.toml and scripts/ was not found")


PROJECT_ROOT = find_project_root()
PROJECT_ROOT

PosixPath('/home/shiawase/ic8_ai/ad_detection')

## Settings

Change `MODEL_PATH` if you want to test another trained model.

In [4]:
MODEL_PATH = PROJECT_ROOT / "models/trained/yolo11m_pretrained_img960_b10_antifp_full_v1/best.pt"
VIDEO_PATH = PROJECT_ROOT / "test.mp4"

RUN_NAME = f"{MODEL_PATH.parent.name}_test"
OUTPUT_DIR = PROJECT_ROOT / "output_video" / RUN_NAME
OUTPUT_VIDEO = OUTPUT_DIR / "test_pred.mp4"
OUTPUT_CSV = OUTPUT_DIR / "detections.csv"

args_path = MODEL_PATH.parent / "args.yaml"
train_args = yaml.safe_load(args_path.read_text()) if args_path.exists() else {}

IMGSZ = int(train_args.get("imgsz", 960))
DETECT_CONF = 0.25
DISPLAY_CONF = 0.55
IOU = 0.50
FRAME_STRIDE = 1
DEVICE = 0 if torch.cuda.is_available() else "cpu"
HALF = DEVICE != "cpu"
LINE_WIDTH = 2

print(f"project: {PROJECT_ROOT}")
print(f"model:   {MODEL_PATH}")
print(f"video:   {VIDEO_PATH}")
print(f"output:  {OUTPUT_DIR}")
print(f"imgsz={IMGSZ}, detect_conf={DETECT_CONF}, display_conf={DISPLAY_CONF}, iou={IOU}")
print(f"device={DEVICE}, half={HALF}, frame_stride={FRAME_STRIDE}")

project: /home/shiawase/ic8_ai/ad_detection
model:   /home/shiawase/ic8_ai/ad_detection/models/trained/yolo11m_pretrained_img960_b10_antifp_full_v1/best.pt
video:   /home/shiawase/ic8_ai/ad_detection/test.mp4
output:  /home/shiawase/ic8_ai/ad_detection/output_video/yolo11m_pretrained_img960_b10_antifp_full_v1_test
imgsz=960, detect_conf=0.25, display_conf=0.55, iou=0.5
device=0, half=True, frame_stride=1


In [5]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(MODEL_PATH)
if not VIDEO_PATH.exists():
    raise FileNotFoundError(VIDEO_PATH)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration_sec = frame_count / fps if fps else 0
cap.release()

print(f"frames: {frame_count}")
print(f"fps:    {fps:.2f}")
print(f"size:   {width}x{height}")
print(f"time:   {duration_sec / 60:.1f} min")

frames: 11182
fps:    29.99
size:   720x1280
time:   6.2 min


## Run inference

This cell can take a while. It writes `test_pred.mp4` and `detections.csv` into `output_video/...`.

In [6]:
model = YOLO(str(MODEL_PATH))

cap = cv2.VideoCapture(str(VIDEO_PATH))
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(OUTPUT_VIDEO), fourcc, fps, (width, height))
if not writer.isOpened():
    cap.release()
    raise RuntimeError(f"Could not create output video: {OUTPUT_VIDEO}")

rows: list[dict] = []
frame_idx = 0
t0 = time.time()

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        result = model.predict(
            frame,
            imgsz=IMGSZ,
            conf=DETECT_CONF,
            iou=IOU,
            device=DEVICE,
            half=HALF,
            verbose=False,
        )[0]

        annotated = frame.copy()

        if result.boxes is not None and len(result.boxes) > 0:
            xyxy = result.boxes.xyxy.detach().cpu().numpy()
            confs = result.boxes.conf.detach().cpu().numpy()
            classes = result.boxes.cls.detach().cpu().numpy().astype(int)

            for det_idx, (box, score, class_id) in enumerate(zip(xyxy, confs, classes)):
                x1, y1, x2, y2 = box.tolist()
                shown = float(score) >= DISPLAY_CONF
                if shown:
                    label = f"{result.names.get(int(class_id), str(class_id))} {float(score):.2f}"
                    p1 = (int(round(x1)), int(round(y1)))
                    p2 = (int(round(x2)), int(round(y2)))
                    cv2.rectangle(annotated, p1, p2, (0, 0, 255), LINE_WIDTH)
                    text_size, baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                    tx1, ty1 = p1
                    ty1 = max(ty1, text_size[1] + baseline + 4)
                    cv2.rectangle(
                        annotated,
                        (tx1, ty1 - text_size[1] - baseline - 4),
                        (tx1 + text_size[0] + 6, ty1),
                        (0, 0, 255),
                        -1,
                    )
                    cv2.putText(
                        annotated,
                        label,
                        (tx1 + 3, ty1 - baseline - 2),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (255, 255, 255),
                        2,
                        cv2.LINE_AA,
                    )
                rows.append(
                    {
                        "frame": frame_idx,
                        "time_sec": frame_idx / fps if fps else 0,
                        "det_idx": det_idx,
                        "class_id": int(class_id),
                        "class_name": result.names.get(int(class_id), str(class_id)),
                        "conf": float(score),
                        "shown": shown,
                        "x1": float(x1),
                        "y1": float(y1),
                        "x2": float(x2),
                        "y2": float(y2),
                        "width": float(x2 - x1),
                        "height": float(y2 - y1),
                    }
                )

        writer.write(annotated)

        frame_idx += 1
        if frame_idx % 100 == 0:
            elapsed = time.time() - t0
            fps_proc = frame_idx / elapsed if elapsed else 0
            print(f"processed {frame_idx}/{frame_count} frames, {fps_proc:.2f} fps")
finally:
    cap.release()
    writer.release()

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

elapsed = time.time() - t0
print(f"Done in {elapsed / 60:.1f} min")
print(f"Processed frames: {frame_idx}")
print(f"Detections:       {len(df)}")
print(f"Video:            {OUTPUT_VIDEO}")
print(f"CSV:              {OUTPUT_CSV}")

processed 100/11182 frames, 25.25 fps
processed 200/11182 frames, 36.55 fps
processed 300/11182 frames, 43.46 fps
processed 400/11182 frames, 48.54 fps
processed 500/11182 frames, 52.11 fps
processed 600/11182 frames, 53.91 fps
processed 700/11182 frames, 55.47 fps
processed 800/11182 frames, 56.61 fps
processed 900/11182 frames, 57.54 fps
processed 1000/11182 frames, 58.44 fps
processed 1100/11182 frames, 54.74 fps
processed 1200/11182 frames, 55.78 fps
processed 1300/11182 frames, 56.69 fps
processed 1400/11182 frames, 57.49 fps
processed 1500/11182 frames, 57.94 fps
processed 1600/11182 frames, 58.35 fps
processed 1700/11182 frames, 58.76 fps
processed 1800/11182 frames, 59.09 fps
processed 1900/11182 frames, 59.58 fps
processed 2000/11182 frames, 60.13 fps
processed 2100/11182 frames, 60.61 fps
processed 2200/11182 frames, 60.89 fps
processed 2300/11182 frames, 60.98 fps
processed 2400/11182 frames, 61.25 fps
processed 2500/11182 frames, 61.53 fps
processed 2600/11182 frames, 61.80

## Detection summary

In [7]:
if df.empty:
    print("No detections")
else:
    display(df.head())
    display(
        df.groupby("class_name")
        .agg(
            detections=("conf", "size"),
            frames=("frame", "nunique"),
            mean_conf=("conf", "mean"),
            max_conf=("conf", "max"),
        )
        .sort_values("detections", ascending=False)
    )

,frame,time_sec,det_idx,class_id,class_name,conf,shown,x1,y1,x2,y2,width,height
0,59,1.967143,0,0,ad_object,0.295410,False,0.000000,489.333344,602.000000,1280.0,602.000000,790.666656
1,60,2.000485,0,0,ad_object,0.323486,False,0.000000,485.000000,565.333374,1280.0,565.333374,795.000000
2,61,2.033826,0,0,ad_object,0.514648,False,514.000000,542.333374,720.000000,1280.0,206.000000,737.666626
3,62,2.067167,0,0,ad_object,0.259766,False,492.666687,521.666687,720.000000,1280.0,227.333313,758.333313
4,63,2.100509,0,0,ad_object,0.445557,False,494.000000,541.000000,720.000000,1280.0,226.000000,739.000000


,detections,frames,mean_conf,max_conf
class_name,,,,
ad_object,5172,3181,0.764714,0.963379
